# 🚀 StoryDiffusion x Agentic Comic Generator
Dùng **LangGraph Agents** từ repo hiện tại để viết kịch bản,
và **Consistent Self-Attention (SDXL)** để vẽ ảnh đồng nhất khuôn mặt.

> Bật **GPU T4/A100** trước khi chạy.

In [ ]:
!pip install -qU diffusers transformers accelerate langchain langchain-openai langgraph pydantic
import sys, os
sys.path.insert(0, '/content/Comic_Studio')  # root của repo
print('✅ Cài xong dependencies')

In [ ]:
os.environ['OPENAI_API_KEY'] = 'sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'  # ← điền key vào đây

from studio_graph.agents import StudioState, run_director, run_writer, run_storyboarder, run_validator
print('✅ Loaded agents thành công!')

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, DDIMScheduler
import sd_utils.story_attention as sa
from sd_utils.story_attention import setup_storydiffusion_state

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SD_MODEL = 'SG161222/RealVisXL_V4.0'
HEIGHT, WIDTH = 768, 768
ID_LENGTH = 3

print('⏳ Loading SDXL...')
pipe = StableDiffusionXLPipeline.from_pretrained(SD_MODEL, torch_dtype=torch.float16, use_safetensors=True)
pipe = pipe.to(device)
pipe.enable_freeu(s1=0.6, s2=0.4, b1=1.1, b2=1.2)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
print('✅ Pipeline ready!')

In [ ]:
import matplotlib.pyplot as plt
from langgraph.graph import StateGraph, END

def run_storydiffusion_renderer(state: StudioState) -> StudioState:
    print('\n[STORY-DIFFUSION RENDERER] Bắt đầu vẽ...')
    schema = state['current_schema']
    panels = schema.get('panels', [])
    prompts = [p.get('panel_prompt_en', p.get('description', '')) for p in panels]
    if not prompts:
        print('Không có panels!')
        return state

    gen = lambda seed: torch.Generator(device=device).manual_seed(seed)

    # ── PASS 1: Ghi nhớ khuôn mặt (write=True) ──
    setup_storydiffusion_state(pipe, id_length=ID_LENGTH, sa32=0.5, sa64=0.5, height=HEIGHT, width=WIDTH)
    sa.write = True
    id_prompts = [prompts[0]] * ID_LENGTH
    print(f'🖌 [PASS 1] Khởi tạo {ID_LENGTH} ID images...')
    pipe(prompt=id_prompts, height=HEIGHT, width=WIDTH,
         num_inference_steps=25, generator=gen(42), guidance_scale=5.0)

    # ── PASS 2: Vẽ từng panel (write=False) ──
    sa.write = False
    out_images = []
    print(f'🎨 [PASS 2] Vẽ {len(prompts)} panels...')
    for i, p in enumerate(prompts):
        sa.attn_count = 0
        sa.cur_step = 0
        img = pipe(prompt=p, height=HEIGHT, width=WIDTH,
                   num_inference_steps=25, generator=gen(42 + i), guidance_scale=5.0).images[0]
        out_images.append(img)
        print(f'  ✔ Panel {i+1}/{len(prompts)}')

    # ── Hiển thị ──
    fig, axes = plt.subplots(1, len(out_images), figsize=(6 * len(out_images), 6))
    if len(out_images) == 1: axes = [axes]
    for ax, img, p in zip(axes, out_images, prompts):
        ax.imshow(img); ax.axis('off')
        ax.set_title(p[:40] + '...', fontsize=8)
    plt.tight_layout(); plt.show()

    state['current_page_idx'] = state.get('current_page_idx', 0) + 1
    return state

# ── Compile Graph ──
workflow = StateGraph(StudioState)
workflow.add_node('director', run_director)
workflow.add_node('writer', run_writer)
workflow.add_node('storyboarder', run_storyboarder)
workflow.add_node('validator', run_validator)
workflow.add_node('renderer', run_storydiffusion_renderer)
workflow.set_entry_point('director')
workflow.add_edge('director', 'writer')
workflow.add_edge('writer', 'storyboarder')
workflow.add_edge('storyboarder', 'validator')
def validator_router(state): return state['next_step']
workflow.add_conditional_edges('validator', validator_router, {'renderer': 'renderer', 'storyboarder': 'storyboarder'})
def renderer_router(state):
    return 'end' if state.get('current_page_idx', 0) >= len(state.get('page_scripts', [])) else 'storyboarder'
workflow.add_conditional_edges('renderer', renderer_router, {'end': END, 'storyboarder': 'storyboarder'})
app = workflow.compile()
print('✅ Graph compiled!')

In [ ]:
idea = 'Một nữ thợ săn tiền thưởng tóc đỏ trên hành tinh hoang dã săn quái vật ngoài không gian'

print('🚀 Bắt đầu tạo truyện...')
config = {'configurable': {'thread_id': 'colab_run_1'}}
for output in app.stream({'user_prompt': idea}, config=config, stream_mode='values'):
    pass